In [ ]:
# Save diagnostics
diagnostics = {
    "total_patches": len(patches),
    "patches_placed": stitch_result.num_patches_placed(),
    "coverage": stitch_result.diagnostics.get('coverage', 0),
    "canvas_width": stitch_result.canvas_width,
    "canvas_height": stitch_result.canvas_height,
    "questions_total": len(answered_questions),
    "questions_attempted": attempted,
    "questions_abstained": abstained,
    "runtime_sec": stitch_result.diagnostics.get('runtime_sec', 0),
}
save_diagnostic_report(diagnostics, config.OUTPUT_DIR / "diagnostics.txt")

# Print final summary
print("\n" + "=" * 60)
print("PIPELINE EXECUTION SUMMARY")
print("=" * 60)
print(f"Input:  {len(patches)} patches → {len(questions)} questions")
print(f"Output: {config.OUTPUT_DIR / 'submission.csv'}")
print(f"")
print(f"Stitching:")
print(f"  - Patches placed: {stitch_result.num_patches_placed()}/{len(patches)}")
print(f"  - Coverage: {stitch_result.diagnostics.get('coverage', 0):.1%}")
print(f"  - Canvas: {stitch_result.canvas_width} × {stitch_result.canvas_height}")
print(f"")
print(f"QA:")
print(f"  - Questions answered: {attempted}/{len(answered_questions)}")
print(f"  - Questions abstained: {abstained}/{len(answered_questions)}")
print(f"")
print(f"Files generated:")
print(f"  - {config.OUTPUT_DIR / 'submission.csv'}")
print(f"  - {config.OUTPUT_DIR / 'stitched_map.png'}")
print(f"  - {config.OUTPUT_DIR / 'diagnostics.txt'}")
print("=" * 60)

## Step 6: Results & Diagnostics

In [ ]:
# Generate and write submission
logger.info("Generating submission...")
submission_rows = generate_submission(answered_questions)

submission_path = write_submission_csv(submission_rows, config.OUTPUT_DIR / "submission.csv")
logger.info(f"✓ Submission written to: {submission_path}")

# Validate submission format
print(f"\nSubmission Format:")
print("-" * 60)
df_submission = pd.read_csv(submission_path)
print(df_submission)
print(f"\nValidation:")
print(f"  Rows: {len(df_submission)}")
print(f"  Columns: {list(df_submission.columns)}")
print(f"  Option value range: {df_submission['option'].min()} - {df_submission['option'].max()}")
print(f"  Option value counts:")
print(df_submission['option'].value_counts().sort_index())

## Step 5: Generate Submission

In [ ]:
# Run QA engine
logger.info("Analyzing questions and generating answers...")
answered_questions = analyze_and_answer_questions(questions, stitch_result.stitched_image)

# Count answers
attempted = sum(1 for q in answered_questions if q.predicted_option != 5)
abstained = len(answered_questions) - attempted

logger.info(f"✓ QA complete!")
logger.info(f"  Questions answered: {attempted}/{len(answered_questions)}")
logger.info(f"  Questions abstained: {abstained}")

# Display answers
print(f"\nPredicted Answers:")
print("-" * 60)
for q in answered_questions:
    if q.predicted_option == 5:
        print(f"{q.question_id}: ABSTAINED (confidence: {q.confidence:.2f})")
    else:
        option_text = q.options[q.predicted_option - 1] if 1 <= q.predicted_option <= len(q.options) else "???"
        print(f"{q.question_id}: Option {q.predicted_option} - {option_text} (confidence: {q.confidence:.2f})")

## Step 4: Answer Questions

In [ ]:
# Run stitching engine
logger.info("Starting stitching engine...")
stitcher = ImageStitcher(patches)
stitch_result = stitcher.stitch()

if stitch_result.success:
    logger.info(f"✓ Stitching successful!")
    logger.info(f"  Patches placed: {stitch_result.num_patches_placed()}/{len(patches)}")
    logger.info(f"  Canvas size: {stitch_result.canvas_width} × {stitch_result.canvas_height}")
    logger.info(f"  Coverage: {stitch_result.diagnostics.get('coverage', 0):.1%}")
    
    # Save stitched image
    stitch_path = save_stitched_image(stitch_result.stitched_image, "stitched_map.png")
    logger.info(f"  Saved to: {stitch_path}")
else:
    logger.error(f"✗ Stitching failed: {stitch_result.error_message}")

# Display stitched map
print(f"\nStitched Map Statistics:")
print(f"  Size: {stitch_result.stitched_image.shape}")
print(f"  Data type: {stitch_result.stitched_image.dtype}")
print(f"  Coverage: {stitch_result.diagnostics.get('coverage', 0):.1%}")

## Step 3: Stitch Patches

In [ ]:
# Load test questions and patches
logger.info("Loading test data...")
questions = load_test_csv(config.TEST_CSV)
patches = load_patches(config.PATCHES_DIR)

logger.info(f"Loaded {len(questions)} questions")
logger.info(f"Loaded {len(patches)} patches")

# Display questions
print(f"\nTest Questions ({len(questions)} total):")
print("-" * 60)
for q in questions[:3]:  # Show first 3
    print(f"ID: {q.question_id}")
    print(f"Question: {q.question_text}")
    print(f"Options: {q.options}")
    print()

# Display patch info
print(f"\nPatch Information:")
print("-" * 60)
print(f"Total patches: {len(patches)}")
if config.PATCH_ANCHOR_ID in patches:
    anchor_patch = patches[config.PATCH_ANCHOR_ID]
    h, w, _ = anchor_patch.shape()
    print(f"Patch shape: {h} × {w}")
    print(f"Estimated grid: {int(np.sqrt(len(patches)))} × {int(np.sqrt(len(patches)))}")

## Step 2: Load Test Data

In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Set random seed for reproducibility
np.random.seed(config.RANDOM_SEED)

# Create output directory
config.OUTPUT_DIR.mkdir(exist_ok=True)

logger.info("=" * 60)
logger.info("GNR Geospatial Stitching Pipeline - Notebook Session")
logger.info("=" * 60)

# Display configuration
print(f"Input directory: {config.DATA_DIR}")
print(f"Output directory: {config.OUTPUT_DIR}")
print(f"Feature detector: {config.FEATURE_DETECTOR}")
print(f"OCR engine: {config.OCR_ENGINE}")
print(f"Abstention threshold: {config.CONFIDENCE_THRESHOLD_ABSTAIN}")
print(f"Random seed: {config.RANDOM_SEED}")

## Step 1: Setup & Configuration

In [ ]:
import sys
from pathlib import Path
import logging
import time

# Add src to path
sys.path.insert(0, str(Path.cwd()))

# Import pipeline modules
from src import config
from src.io_utils import (
    load_test_csv, load_patches, write_submission_csv, 
    save_stitched_image, save_diagnostic_report
)
from src.stitching import ImageStitcher
from src.qa import analyze_and_answer_questions
from src.submission import generate_submission

# Standard libraries
import numpy as np
import cv2
import pandas as pd

print("✓ All imports successful")

# GNR Geospatial Image Stitching & Analysis Pipeline

This notebook reconstructs a large geospatial map from shuffled, overlapping image patches and answers multiple-choice questions based on the reconstructed map.

**Pipeline:**
1. **Stitching**: Feature matching (SIFT/ORB) + global placement
2. **OCR**: Text extraction from stitched map
3. **QA**: Answer MCQ questions based on visual + textual analysis
4. **Submission**: Generate competition-valid predictions with strategic abstention